### Task 1: Parquet Internals and Metadata Inspection

**Step 1:** Create a dataset with 500,000 rows using the following schema (you will reuse this dataset throughout the lab). Use `np.random.seed(42)` for reproducibility.

| Column | Type | Description |
|---|---|---|
| `user_id` | int | Unique user identifier (1 to 500,000) |
| `city` | string | One of 10 city names (e.g., "Berlin", "Tokyo", "New York", etc.) |
| `score` | float | Random score between 0 and 100 |
| `active` | bool | Whether the user is active |
| `signup_date` | timestamp | Random date within the last 3 years |
| `age` | int | Random age between 18 and 80 |
| `sessions` | int | Number of sessions (random, 0 to 500) |
| `revenue` | float | Revenue in dollars (random, 0 to 1000) |

In [1]:
import os
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

np.random.seed(42)
n = 500_000

cities = ["Berlin", "Tokyo", "New York", "Seoul", "Nairobi", "Toronto", "Lima", "Paris", "London", "Sydney"]

# Use a fixed "today" for reproducibility (matches our current course date context)
end_date = pd.Timestamp("2026-03-04")
days_back = np.random.randint(0, 365 * 3 + 1, size=n)  # within last 3 years
signup_date = end_date - pd.to_timedelta(days_back, unit="D")

df = pd.DataFrame({
    "user_id": np.arange(1, n + 1, dtype=np.int64),
    "city": np.random.choice(cities, size=n),
    "score": np.random.uniform(0, 100, size=n),
    "active": np.random.choice([True, False], size=n),
    "signup_date": signup_date,
    "age": np.random.randint(18, 81, size=n),          # 18..80 inclusive
    "sessions": np.random.randint(0, 501, size=n),     # 0..500 inclusive
    "revenue": np.random.uniform(0, 1000, size=n),
})

df.head()

,user_id,city,score,active,signup_date,age,sessions,revenue
0,1,Berlin,89.576945,False,2023-10-26,32,456,542.163649
1,2,Toronto,73.575191,True,2023-03-05,69,440,800.395595
2,3,Tokyo,23.354016,False,2023-04-25,78,327,976.588795
3,4,London,17.958558,True,2025-11-03,37,248,448.556329
4,5,Paris,2.642914,False,2024-11-23,73,420,816.764634


**Step 2:** Save the dataset as a Parquet file. Then use `pyarrow.parquet.ParquetFile` to inspect:
- The number of row groups
- The number of rows and columns
- The schema (column names and types)
- For each column in the first row group: physical type, compressed size, and any available statistics (min, max, null count)

In [2]:
import pyarrow as pa
import pyarrow.parquet as pq

parquet_path = "users_500k.parquet"

# Write to Parquet
df.to_parquet(parquet_path, index=False)

# Inspect metadata
parquet_file = pq.ParquetFile(parquet_path)

print("=== Parquet File Metadata ===")
print(f"Number of row groups: {parquet_file.metadata.num_row_groups}")
print(f"Number of rows:      {parquet_file.metadata.num_rows}")
print(f"Number of columns:   {parquet_file.metadata.num_columns}")

print("\n=== Schema (Arrow) ===")
print(parquet_file.schema_arrow)

# First row group column details
rg0 = parquet_file.metadata.row_group(0)

print("\n=== First Row Group: Column Details ===")
for i in range(rg0.num_columns):
    col = rg0.column(i)
    stats = col.statistics  # may be None depending on writer/settings
    
    min_v = getattr(stats, "min", None) if stats else None
    max_v = getattr(stats, "max", None) if stats else None
    nulls = getattr(stats, "null_count", None) if stats else None
    
    print(
        f"Column: {col.path_in_schema:12s} | "
        f"Physical: {str(col.physical_type):10s} | "
        f"Compressed: {col.total_compressed_size:>10d} bytes | "
        f"min={min_v} | max={max_v} | nulls={nulls}"
    )

=== Parquet File Metadata ===
Number of row groups: 1
Number of rows:      500000
Number of columns:   8

=== Schema (Arrow) ===
user_id: int64
city: string
score: double
active: bool
signup_date: timestamp[ns]
age: int32
sessions: int32
revenue: double
-- schema metadata --
pandas: '{"index_columns": [], "column_indexes": [], "columns": [{"name":' + 974

=== First Row Group: Column Details ===
Column: user_id      | Physical: INT64      | Compressed:    2280081 bytes | min=1 | max=500000 | nulls=0
Column: city         | Physical: BYTE_ARRAY | Compressed:     251171 bytes | min=Berlin | max=Toronto | nulls=0
Column: score        | Physical: DOUBLE     | Compressed:    4279334 bytes | min=0.00022849935341628935 | max=99.99943752472724 | nulls=0
Column: active       | Physical: BOOLEAN    | Compressed:      62553 bytes | min=False | max=True | nulls=0
Column: signup_date  | Physical: INT64      | Compressed:     697300 bytes | min=2023-03-05 00:00:00 | max=2026-03-04 00:00:00 | nulls=0
C

In [3]:
# First row group column details
rg0 = parquet_file.metadata.row_group(0)

print("\n=== First Row Group: Column Details ===")
for i in range(rg0.num_columns):
    col = rg0.column(i)
    stats = col.statistics  # may be None depending on writer/settings
    
    min_v = getattr(stats, "min", None) if stats else None
    max_v = getattr(stats, "max", None) if stats else None
    nulls = getattr(stats, "null_count", None) if stats else None
    
    print(
        f"Column: {col.path_in_schema:12s} | "
        f"Physical: {str(col.physical_type):10s} | "
        f"Compressed: {col.total_compressed_size:>10d} bytes | "
        f"min={min_v} | max={max_v} | nulls={nulls}"
    )


=== First Row Group: Column Details ===
Column: user_id      | Physical: INT64      | Compressed:    2280081 bytes | min=1 | max=500000 | nulls=0
Column: city         | Physical: BYTE_ARRAY | Compressed:     251171 bytes | min=Berlin | max=Toronto | nulls=0
Column: score        | Physical: DOUBLE     | Compressed:    4279334 bytes | min=0.00022849935341628935 | max=99.99943752472724 | nulls=0
Column: active       | Physical: BOOLEAN    | Compressed:      62553 bytes | min=False | max=True | nulls=0
Column: signup_date  | Physical: INT64      | Compressed:     697300 bytes | min=2023-03-05 00:00:00 | max=2026-03-04 00:00:00 | nulls=0
Column: age          | Physical: INT32      | Compressed:     376346 bytes | min=18 | max=80 | nulls=0
Column: sessions     | Physical: INT32      | Compressed:     565609 bytes | min=0 | max=500 | nulls=0
Column: revenue      | Physical: DOUBLE     | Compressed:    4279334 bytes | min=0.0009958058022618843 | max=999.9978869129644 | nulls=0


**Step 3:** Save the same dataset as CSV. Compare file sizes. Print both sizes in KB and the compression ratio.

In [4]:
csv_path = "users_500k.csv"
df.to_csv(csv_path, index=False)

parquet_size_bytes = os.path.getsize(parquet_path)
csv_size_bytes = os.path.getsize(csv_path)

parquet_kb = parquet_size_bytes / 1024
csv_kb = csv_size_bytes / 1024

compression_ratio = csv_size_bytes / parquet_size_bytes  # "how many times larger CSV is"

print("=== File Size Comparison ===")
print(f"Parquet size: {parquet_kb:,.1f} KB")
print(f"CSV size:     {csv_kb:,.1f} KB")
print(f"Compression ratio (CSV / Parquet): {compression_ratio:.2f}x")

=== File Size Comparison ===
Parquet size: 12,496.1 KB
CSV size:     36,280.6 KB
Compression ratio (CSV / Parquet): 2.90x


**Step 4:** Write a markdown cell explaining what information the Parquet metadata provides that CSV does not, and why that matters for query performance.

### What Parquet metadata provides that CSV does not (and why it matters)
Parquet stores a footer with **schema** (column names + types), **row group layout**, and often **per-column statistics** (like min, max, and null counts) for each row group/column chunk. This means an engine can **skip reading entire row groups** when a filter can’t match (predicate pushdown / row-group pruning) and can **read only the required columns** (column pruning) without scanning everything.

CSV does not have this structured metadata: it has no built-in types, no row groups, and no column-level statistics, so most queries require **reading and parsing the entire file** even if you only need a few columns or a small filtered subset. This is why Parquet is usually much faster and smaller for analytics.

### Task 2: Column Projection and Selective Reads

**Step 1:** Read the full Parquet file from Task 1 and time it.

In [5]:
import pandas as pd

parquet_path = "users_500k.parquet"
csv_path = "users_500k.csv"

%time full_parquet = pd.read_parquet(parquet_path)
full_parquet.shape

CPU times: total: 219 ms
Wall time: 85.2 ms


(500000, 8)

**Step 2:** Read only 2 columns from the same Parquet file and time it.

In [6]:
%time two_cols_parquet = pd.read_parquet(parquet_path, columns=["user_id", "city"])
two_cols_parquet.shape

CPU times: total: 46.9 ms
Wall time: 44.7 ms


(500000, 2)

**Step 3:** Read only 2 columns from the CSV file and time it (you will need to read the entire CSV and select columns after, which is what pandas does).

In [7]:
%time full_csv = pd.read_csv(csv_path)          # pandas must parse everything
%time two_cols_csv = full_csv[["user_id", "city"]]
two_cols_csv.shape

CPU times: total: 391 ms
Wall time: 424 ms
CPU times: total: 0 ns
Wall time: 6.07 ms


(500000, 2)

In [8]:
%%time
two_cols_csv_combined = pd.read_csv(csv_path)[["user_id", "city"]]
two_cols_csv_combined.shape

CPU times: total: 375 ms
Wall time: 382 ms


(500000, 2)

**Step 4:** Write a markdown cell explaining why Parquet selective reads are faster. Connect your explanation to the column-chunk layout you observed in Task 1.

### Why Parquet selective reads are faster

Parquet stores data in **row groups**, and inside each row group it stores each column separately as a **column chunk**. When we call `pd.read_parquet(..., columns=["user_id","city"])`, the reader can jump directly to just the `user_id` and `city` column chunks and **skip reading the other columns entirely**. That reduces disk I/O and also reduces decompression work because only those two columns’ chunks are decompressed.

CSV is row-oriented text: values from all columns are interleaved on every line. Even if we only want 2 columns, pandas still has to **read and parse every row and every column** to understand the file, then we drop the extra columns afterward. So Parquet is faster for selective reads because its **column-chunk layout** supports true column pruning, while CSV does not.

### Task 3: Predicate Pushdown in Practice

**Step 1:** Using PyArrow, read the Parquet file with a filter (e.g., `age > 50`). Time the read.

In [9]:
import pyarrow.parquet as pq

parquet_path = "users_500k.parquet"

# Predicate pushdown happens here (Parquet scan can skip row groups using stats)
%time arrow_filtered = pq.read_table(parquet_path, filters=[("age", ">", 50)])
print("PyArrow filtered rows:", arrow_filtered.num_rows)
arrow_filtered.to_pandas().head()

CPU times: total: 109 ms
Wall time: 40.5 ms
PyArrow filtered rows: 237773


,user_id,city,score,active,signup_date,age,sessions,revenue
0,2,Toronto,73.575191,True,2023-03-05,69,440,800.395595
1,3,Tokyo,23.354016,False,2023-04-25,78,327,976.588795
2,5,Paris,2.642914,False,2024-11-23,73,420,816.764634
3,7,Seoul,36.161321,False,2025-12-07,70,194,916.677209
4,9,Seoul,92.452581,True,2025-10-25,65,25,330.268020


**Step 2:** Read the full Parquet file without a filter and apply the same filter in pandas after loading. Time this approach.

In [10]:
import pandas as pd

%time df_full = pd.read_parquet(parquet_path)
%time pandas_filtered = df_full[df_full["age"] > 50]
print("pandas filtered rows:", len(pandas_filtered))

CPU times: total: 125 ms
Wall time: 53.3 ms
CPU times: total: 15.6 ms
Wall time: 12 ms
pandas filtered rows: 237773


**Step 3:** Compare the number of rows returned and the execution times. Write a markdown cell explaining what predicate pushdown does and why it is faster.

In [11]:
# Compare row counts (should match)
arrow_rows = arrow_filtered.num_rows
pandas_rows = len(pandas_filtered)

print("Row count match?", arrow_rows == pandas_rows)
print("PyArrow rows:", arrow_rows)
print("pandas rows:", pandas_rows)

Row count match? True
PyArrow rows: 237773
pandas rows: 237773


### Predicate pushdown: what it does and why it’s faster

Predicate pushdown means the filter condition (e.g., `age > 50`) is applied **as early as possible**, ideally while scanning the Parquet file. Parquet stores per-row-group statistics (like min/max for columns) in its metadata, so the reader can **skip entire row groups** whose `age` range cannot satisfy the predicate. That reduces disk I/O and decompression because those row groups are never read.

The pandas “load everything then filter” approach must first read and decode the **entire dataset** into memory, and only then drop rows. Pushdown is faster because it avoids reading data that will be discarded anyway.

**Step 4:** Run the same filtered query using DuckDB's SQL interface directly on the Parquet file:

```python
import duckdb
result = duckdb.sql("SELECT * FROM 'your_file.parquet' WHERE age > 50").df()
```

Time it and compare with both PyArrow and pandas approaches.

In [12]:
import duckdb

# DuckDB scans Parquet directly and can push filters down into the scan
%time duckdb_result = duckdb.sql(f"SELECT * FROM '{parquet_path}' WHERE age > 50").df()
print("DuckDB filtered rows:", len(duckdb_result))

CPU times: total: 78.1 ms
Wall time: 64.2 ms
DuckDB filtered rows: 237773


### Task 4: pandas vs DuckDB — Identical Queries

**Query 1:** Count records per city.

In [13]:
import pandas as pd
import duckdb

df = pd.read_parquet("users_500k.parquet")

In [14]:
%time pandas_q1 = df.groupby("city").size().reset_index(name="count")
pandas_q1

CPU times: total: 31.2 ms
Wall time: 30.3 ms


,city,count
0,Berlin,49928
1,Lima,49968
2,London,50199
3,Nairobi,50274
4,New York,49959
5,Paris,49868
6,Seoul,49894
7,Sydney,49841
8,Tokyo,50071
9,Toronto,49998


In [15]:
%time duckdb_q1 = duckdb.sql("SELECT city, COUNT(*) AS count FROM df GROUP BY city").df()
duckdb_q1

CPU times: total: 78.1 ms
Wall time: 10.7 ms


,city,count
0,Tokyo,50071
1,Paris,49868
2,Sydney,49841
3,Lima,49968
4,Berlin,49928
5,Nairobi,50274
6,London,50199
7,Toronto,49998
8,New York,49959
9,Seoul,49894


**Query 2:** Average score by city, ordered by average score descending.

In [16]:
%time pandas_q2 = df.groupby("city")["score"].mean().reset_index(name="avg_score").sort_values("avg_score", ascending=False)
pandas_q2

CPU times: total: 31.2 ms
Wall time: 30.3 ms


,city,avg_score
8,Tokyo,50.229611
5,Paris,50.107980
2,London,50.104380
7,Sydney,50.057056
4,New York,50.046353
6,Seoul,50.044812
1,Lima,49.959550
0,Berlin,49.929480
3,Nairobi,49.913170
9,Toronto,49.905492


In [17]:
%time duckdb_q2 = duckdb.sql("SELECT city, AVG(score) AS avg_score FROM df GROUP BY city ORDER BY avg_score DESC").df()
duckdb_q2

CPU times: total: 93.8 ms
Wall time: 12 ms


,city,avg_score
0,Tokyo,50.229611
1,Paris,50.107980
2,London,50.104380
3,Sydney,50.057056
4,New York,50.046353
5,Seoul,50.044812
6,Lima,49.959550
7,Berlin,49.929480
8,Nairobi,49.913170
9,Toronto,49.905492


**Query 3:** For each city, find the percentage of active users whose score is above 75.

In [18]:
%time pandas_q3 = df[df["active"]].groupby("city").apply(lambda x: (x["score"] > 75).mean() * 100, include_groups=False).reset_index(name="pct_active_score_above_75")
pandas_q3

CPU times: total: 62.5 ms
Wall time: 53.8 ms


,city,pct_active_score_above_75
0,Berlin,24.942681
1,Lima,25.064713
2,London,24.747739
3,Nairobi,24.804167
4,New York,25.150922
5,Paris,25.135602
6,Seoul,25.080141
7,Sydney,25.227547
8,Tokyo,25.134985
9,Toronto,24.671393


In [19]:
%time duckdb_q3 = duckdb.sql("SELECT city, AVG(CASE WHEN score > 75 THEN 1 ELSE 0 END) * 100 AS pct_active_score_above_75 FROM df WHERE active = TRUE GROUP BY city").df()
duckdb_q3

CPU times: total: 15.6 ms
Wall time: 11.3 ms


,city,pct_active_score_above_75
0,Seoul,25.080141
1,Tokyo,25.134985
2,Paris,25.135602
3,Toronto,24.671393
4,London,24.747739
5,Lima,25.064713
6,Berlin,24.942681
7,Sydney,25.227547
8,Nairobi,24.804167
9,New York,25.150922


**Query 4:** Find the top 10 users by score for each city (window function / rank).

In [20]:
%time pandas_q4 = df.assign(rank=df.groupby("city")["score"].rank(method="first", ascending=False)).query("rank <= 10").sort_values(["city","rank"])
pandas_q4.head(20)

CPU times: total: 203 ms
Wall time: 212 ms


,user_id,city,score,active,signup_date,age,sessions,revenue,rank
202915,202916,Berlin,99.998237,True,2026-02-15,43,175,780.348053,1.0
395199,395200,Berlin,99.988672,False,2023-04-23,62,231,246.554441,2.0
71636,71637,Berlin,99.988625,False,2024-03-23,25,412,14.809389,3.0
324415,324416,Berlin,99.987230,False,2024-07-21,73,55,240.050492,4.0
287683,287684,Berlin,99.986486,True,2023-11-27,74,347,264.763911,5.0
173816,173817,Berlin,99.985634,False,2025-03-31,24,371,199.255113,6.0
179354,179355,Berlin,99.983774,False,2024-03-22,78,283,444.193323,7.0
353295,353296,Berlin,99.978708,True,2025-07-09,78,280,291.656186,8.0
492373,492374,Berlin,99.977963,False,2024-09-13,72,274,110.801542,9.0
131583,131584,Berlin,99.976629,False,2025-11-08,27,273,386.054454,10.0


In [21]:
%time duckdb_q4 = duckdb.sql("SELECT * FROM (SELECT *, ROW_NUMBER() OVER (PARTITION BY city ORDER BY score DESC) AS rank FROM df) WHERE rank <= 10 ORDER BY city, rank").df()
duckdb_q4.head(20)

CPU times: total: 688 ms
Wall time: 221 ms


,user_id,city,score,active,signup_date,age,sessions,revenue,rank
0,202916,Berlin,99.998237,True,2026-02-15,43,175,780.348053,1
1,395200,Berlin,99.988672,False,2023-04-23,62,231,246.554441,2
2,71637,Berlin,99.988625,False,2024-03-23,25,412,14.809389,3
3,324416,Berlin,99.987230,False,2024-07-21,73,55,240.050492,4
4,287684,Berlin,99.986486,True,2023-11-27,74,347,264.763911,5
5,173817,Berlin,99.985634,False,2025-03-31,24,371,199.255113,6
6,179355,Berlin,99.983774,False,2024-03-22,78,283,444.193323,7
7,353296,Berlin,99.978708,True,2025-07-09,78,280,291.656186,8
8,492374,Berlin,99.977963,False,2024-09-13,72,274,110.801542,9
9,131584,Berlin,99.976629,False,2025-11-08,27,273,386.054454,10


**Query 5:** Compute a running total of scores ordered by user_id, partitioned by city.

In [22]:
%time pandas_q5 = df.sort_values(["city","user_id"]).assign(running_total=lambda x: x.groupby("city")["score"].cumsum())
pandas_q5.head()

CPU times: total: 141 ms
Wall time: 151 ms


,user_id,city,score,active,signup_date,age,sessions,revenue,running_total
0,1,Berlin,89.576945,False,2023-10-26,32,456,542.163649,89.576945
7,8,Berlin,84.949780,False,2023-10-15,20,219,646.953382,174.526725
15,16,Berlin,6.580206,False,2024-11-30,67,260,38.501181,181.106931
37,38,Berlin,93.911150,False,2025-04-23,46,359,904.156764,275.018081
39,40,Berlin,44.989325,False,2025-07-06,69,340,263.943550,320.007406


In [23]:
%time duckdb_q5 = duckdb.sql("SELECT *, SUM(score) OVER (PARTITION BY city ORDER BY user_id ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS running_total FROM df").df()
duckdb_q5.head()

CPU times: total: 781 ms
Wall time: 213 ms


,user_id,city,score,active,signup_date,age,sessions,revenue,running_total
0,1,Berlin,89.576945,False,2023-10-26,32,456,542.163649,89.576945
1,8,Berlin,84.949780,False,2023-10-15,20,219,646.953382,174.526725
2,16,Berlin,6.580206,False,2024-11-30,67,260,38.501181,181.106931
3,38,Berlin,93.911150,False,2025-04-23,46,359,904.156764,275.018081
4,40,Berlin,44.989325,False,2025-07-06,69,340,263.943550,320.007406


### pandas vs DuckDB Comparison

In this notebook, both pandas and DuckDB were able to produce the correct results for all five queries. However, there were some differences in ease of use and performance.

For Query 1 (count records per city) and Query 2 (average score by city), both pandas and DuckDB were easy to express and the performance difference was minimal. These are simple aggregation operations, and both tools handle them efficiently.

For Query 3 (percentage of active users with score above 75), both approaches worked well, although the SQL version in DuckDB was slightly clearer because the filtering and aggregation could be written directly in one query.

For Query 4 (top 10 users by score per city) and Query 5 (running total of scores by city), DuckDB was easier to express and more natural to write. SQL window functions such as `ROW_NUMBER()` and `SUM() OVER()` are designed specifically for ranking and running totals. In pandas, these operations required additional steps such as `rank()` and `cumsum()`.

Based on the timings observed in this notebook, the performance difference was small for simple aggregations, but the conceptual difference became more noticeable for more complex analytical queries. In particular, window-function-style operations (Query 4 and Query 5) were clearer and more natural to implement using DuckDB.

### Task 5: Arrow as the Bridge

**Step 1:** Create a pandas DataFrame and convert it to an Arrow Table using `pyarrow.Table.from_pandas()`.

In [24]:
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

sample_df = pd.DataFrame({
    "user_id": [1, 2, 3, 4, 5],
    "city": ["Berlin", "Tokyo", "Lima", "Seoul", "Paris"],
    "score": [88.5, 91.2, 76.4, 95.0, 84.3],
    "active": [True, False, True, True, False]
})

arrow_table = pa.Table.from_pandas(sample_df)

print("Original pandas DataFrame:")
print(sample_df)

print("\nArrow Table:")
print(arrow_table)

Original pandas DataFrame:
   user_id    city  score  active
0        1  Berlin   88.5    True
1        2   Tokyo   91.2   False
2        3    Lima   76.4    True
3        4   Seoul   95.0    True
4        5   Paris   84.3   False

Arrow Table:
pyarrow.Table
user_id: int64
city: string
score: double
active: bool
----
user_id: [[1,2,3,4,5]]
city: [["Berlin","Tokyo","Lima","Seoul","Paris"]]
score: [[88.5,91.2,76.4,95,84.3]]
active: [[true,false,true,true,false]]


**Step 2:** Inspect the Arrow Table schema and compare it to the pandas dtypes. Note any differences.

In [25]:
print("=== pandas dtypes ===")
print(sample_df.dtypes)

print("\n=== Arrow schema ===")
print(arrow_table.schema)

=== pandas dtypes ===
user_id      int64
city        object
score      float64
active        bool
dtype: object

=== Arrow schema ===
user_id: int64
city: string
score: double
active: bool
-- schema metadata --
pandas: '{"index_columns": [{"kind": "range", "name": null, "start": 0, "' + 696


### Comparing pandas dtypes and Arrow schema

The pandas DataFrame uses pandas/NumPy dtypes such as `int64`, `float64`, `bool`, and `object` for strings.  
The Arrow Table shows Arrow-native types such as `int64`, `double`, `bool`, and `string`.  

A key difference is that pandas often stores strings as `object`, while Arrow uses a dedicated `string` type, which is more efficient and better suited for columnar memory operations.

**Step 3:** Write the Arrow Table to Parquet using `pyarrow.parquet.write_table()`. Read it back into a new Arrow Table.

In [26]:
arrow_parquet_path = "arrow_bridge.parquet"

pq.write_table(arrow_table, arrow_parquet_path)

arrow_table_read = pq.read_table(arrow_parquet_path)

print("Arrow Table read back from Parquet:")
print(arrow_table_read)

Arrow Table read back from Parquet:
pyarrow.Table
user_id: int64
city: string
score: double
active: bool
----
user_id: [[1,2,3,4,5]]
city: [["Berlin","Tokyo","Lima","Seoul","Paris"]]
score: [[88.5,91.2,76.4,95,84.3]]
active: [[true,false,true,true,false]]


**Step 4:** Convert the Arrow Table back to a pandas DataFrame using `.to_pandas()`. Verify the data matches the original.

In [27]:
df_from_arrow = arrow_table_read.to_pandas()

print("Converted back to pandas:")
print(df_from_arrow)

print("\nData matches original?:", sample_df.equals(df_from_arrow))

Converted back to pandas:
   user_id    city  score  active
0        1  Berlin   88.5    True
1        2   Tokyo   91.2   False
2        3    Lima   76.4    True
3        4   Seoul   95.0    True
4        5   Paris   84.3   False

Data matches original?: True


**Step 5:** Demonstrate the pandas `dtype_backend="pyarrow"` option by reading the Parquet file with Arrow-backed dtypes. Print the dtypes and compare with traditional pandas dtypes.

In [28]:
df_arrow_backed = pd.read_parquet(arrow_parquet_path, dtype_backend="pyarrow")
df_traditional = pd.read_parquet(arrow_parquet_path)

print("=== Arrow-backed pandas dtypes ===")
print(df_arrow_backed.dtypes)

print("\n=== Traditional pandas dtypes ===")
print(df_traditional.dtypes)

=== Arrow-backed pandas dtypes ===
user_id     int64[pyarrow]
city       string[pyarrow]
score      double[pyarrow]
active       bool[pyarrow]
dtype: object

=== Traditional pandas dtypes ===
user_id      int64
city        object
score      float64
active        bool
dtype: object


**Step 6:** Write a markdown cell explaining the role of Arrow in the modern analytics stack. How does it connect Parquet (disk) to pandas (analysis) to DuckDB (SQL)?

### The role of Arrow in the modern analytics stack

Apache Arrow is the bridge between storage formats and analytics tools. Parquet stores data efficiently on disk in a columnar format, while Arrow represents columnar data efficiently in memory. This means data can move from Parquet into Arrow with minimal overhead.

Arrow also helps tools work together. pandas can convert DataFrames to and from Arrow Tables, and modern pandas can use Arrow-backed dtypes directly. DuckDB can also work efficiently with Arrow-style columnar memory, which helps it query data without unnecessary copying.

In the modern analytics stack, Parquet is the storage layer, Arrow is the in-memory layer, and pandas or DuckDB are execution/analysis layers. Arrow connects them by providing a common columnar format that supports fast reads, efficient memory use, and interoperability across tools.